# 00 — Data Audit, Feature Eligibility & Cohort

**Project:** LendingClub loan default risk prediction (2007–2020 Q3 Feather archive).  
**Prediction goal:** At/near loan origination, predict **Fully Paid (0)** versus **Charged Off (1)** for funded loans. This is an observed resolved-outcome proxy, not a guaranteed lifetime default label.

This notebook answers three questions: **What data do we have? Which fields are eligible? Which loan years/terms can we model?**

- Keep the original Feather file unchanged.
- An unexplained `-1` is **not automatically missing**; it is a value to investigate.
- Keep feature selection and cohort decisions provisional until source timing and maturity are checked.
- Detailed EDA / feature engineering happens in Notebook 01; split and fitted preprocessing in Notebook 02.

**Prior investigation (reference, not a new run):** ~2.93M rows, 141 columns; PyArrow successfully decoded 22 categorical columns that included dictionary-encoded fields causing Polars read errors. The source snapshot date still needs verification.

## 1. Load the original data

**Why:** Confirm we are using the intended file and record its dimensions/types before any analysis.  
**Output:** `lf` (lazy dataset), schema, row count, and a small overview. No data is modified.

In [7]:
from pathlib import Path
import json
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.feather as feather
from IPython.display import display

ROOT = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / "configs/project.json").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project folder or notebooks/ directory.")

config = json.loads((ROOT / "configs/project.json").read_text(encoding="utf-8"))
DATA_PATH = Path(config["raw_data_path"])
if not DATA_PATH.is_absolute():
    DATA_PATH = ROOT / DATA_PATH
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Feather file not found: {DATA_PATH}")

AUDIT_DIR = ROOT / "reports/audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
lf = pl.scan_ipc(str(DATA_PATH))
schema = lf.collect_schema()
columns = schema.names()
n_rows = lf.select(pl.len()).collect().item()

print(f"File: {DATA_PATH.name} | Rows: {n_rows:,} | Columns: {len(columns)}")
display(pl.DataFrame({
    "dtype": [str(schema[c]) for c in columns]
}).group_by("dtype").len().sort("len", descending=True))


File: lending_club_2020_q3.feather | Rows: 2,925,492 | Columns: 141


dtype,len
str,u32
"""Int16""",54
"""Float32""",26
"""Categorical""",22
"""Int32""",15
"""Int64""",15
"""Datetime(time_unit='ns', time_…",9


## 2. One consolidated column audit

**Why:** Detect real nulls/NaNs and the suspicious numeric `-1` sentinel. Earlier Polars scans failed on some dictionary-encoded categorical columns, so this cell reads *categorical columns individually with PyArrow*. It is not necessary to rerun separate recovery, reconciliation, and verification notebooks.

**Output:** one `00_column_audit.csv` with every column. `minus_one_pct` is **suspected placeholder prevalence**, not confirmed effective missingness. No replacements or feature deletions happen here.

In [8]:
num_types = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64
}
cat_types = {pl.String, pl.Categorical, pl.Enum}
cat_cols = [c for c in columns if schema[c].base_type() in cat_types]
other_cols = [c for c in columns if c not in cat_cols]
records = []

# Numeric/date columns: scan in small batches to limit memory use.
for start in range(0, len(other_cols), 20):
    batch = other_cols[start:start + 20]
    exprs = []
    for i, col in enumerate(batch):
        dtype = schema[col].base_type()
        exprs.append(pl.col(col).null_count().alias(f"null_{i}"))
        if dtype in {pl.Float32, pl.Float64}:
            exprs.append(pl.col(col).is_nan().sum().alias(f"nan_{i}"))
        if dtype in num_types:
            exprs.append((pl.col(col) == -1).fill_null(False).sum().alias(f"neg_{i}"))
    values = lf.select(exprs).collect(engine="streaming").row(0, named=True)
    for i, col in enumerate(batch):
        records.append({
            "feature": col, "dtype": str(schema[col]),
            "null_count": int(values[f"null_{i}"]),
            "nan_count": int(values.get(f"nan_{i}", 0)),
            "minus_one_count": int(values[f"neg_{i}"]) if f"neg_{i}" in values else None
        })

# PyArrow correctly handles dictionary-encoded categorical nulls.
for col in cat_cols:
    arr = feather.read_table(str(DATA_PATH), columns=[col]).column(0)
    decoded = pc.dictionary_decode(arr) if pa.types.is_dictionary(arr.type) else arr
    if len(decoded) != n_rows or decoded.null_count != arr.null_count:
        raise ValueError(f"Categorical decoding failed verification for {col}")
    records.append({
        "feature": col, "dtype": str(schema[col]),
        "null_count": int(arr.null_count), "nan_count": 0,
        "minus_one_count": None
    })

audit = pl.DataFrame(records)
assert audit.height == len(columns), "Some columns were not audited."
audit = audit.with_columns(
    (pl.col("null_count") + pl.col("nan_count")).alias("missing_count"),
    (100 * (pl.col("null_count") + pl.col("nan_count")) / n_rows)
        .round(2).alias("missing_pct"),
    (100 * pl.col("minus_one_count") / n_rows)
        .round(2).alias("minus_one_pct")
)

print(f"Audited {audit.height} columns; categorical decoding: {len(cat_cols)}/{len(cat_cols)}")
print("Columns with numeric -1:", audit.filter(pl.col("minus_one_count") > 0).height)
display(audit.sort("missing_pct", descending=True).head(15))
print("Highest -1 prevalence:")
display(audit.sort("minus_one_pct", descending=True, nulls_last=True).head(15))
audit.write_csv(AUDIT_DIR / "00_column_audit.csv")


Audited 141 columns; categorical decoding: 22/22
Columns with numeric -1: 81


feature,dtype,null_count,nan_count,minus_one_count,missing_count,missing_pct,minus_one_pct
str,str,i64,i64,i64,i64,f64,f64
"""hardship_loan_status""","""Categorical""",2782081,0,null,2782081,95.1,null
"""hardship_start_date""","""Datetime(time_unit='ns', time_…",2781854,0,null,2781854,95.09,null
"""hardship_end_date""","""Datetime(time_unit='ns', time_…",2781854,0,null,2781854,95.09,null
"""payment_plan_start_date""","""Datetime(time_unit='ns', time_…",2781854,0,null,2781854,95.09,null
"""hardship_type""","""Categorical""",2781854,0,null,2781854,95.09,null
…,…,…,…,…,…,…,…
"""emp_title""","""Categorical""",264086,0,null,264086,9.03,null
"""emp_length""","""Categorical""",205220,0,null,205220,7.01,null
"""hardship_flag""","""Categorical""",38435,0,null,38435,1.31,null


Highest -1 prevalence:


feature,dtype,null_count,nan_count,minus_one_count,missing_count,missing_pct,minus_one_pct
str,str,i64,i64,i64,i64,f64,f64
"""deferral_term""","""Int16""",0,0,2781854,0,0.0,95.09
"""hardship_length""","""Int16""",0,0,2781854,0,0.0,95.09
"""hardship_dpd""","""Int16""",0,0,2781855,0,0.0,95.09
"""orig_projected_additional_accr…","""Float32""",0,0,2746252,0,0.0,93.87
"""hardship_amount""","""Float32""",0,0,2743416,0,0.0,93.78
…,…,…,…,…,…,…,…
"""sec_app_mort_acc""","""Int16""",0,0,2727668,0,0.0,93.24
"""sec_app_open_acc""","""Int16""",0,0,2727668,0,0.0,93.24
"""sec_app_open_act_il""","""Int16""",0,0,2727668,0,0.0,93.24


### 2.1 — Placeholder Investigation

**Why?** The cleaned LendingClub dataset uses `-1` in several numeric columns. These values may represent missing, unavailable, or inapplicable information rather than actual measurements.

**Objective:** Identify affected features and investigate their meanings before choosing a cohort or preprocessing strategy.

**Method:** Reuse the column audit to identify features containing `-1`. Interpret important features using the data dictionary and their availability across issue years.

**Decision:** Do not automatically replace every `-1` with null. Record validated placeholder rules and apply them during preprocessing.

In [13]:
# 2.1 — Review suspected numeric placeholders

placeholder_review = (
    audit
    .filter(pl.col("minus_one_count") > 0)
    .select(
        "feature",
        "minus_one_count",
        "minus_one_pct",
        "missing_pct"
    )
    .sort("minus_one_pct", descending=True)
)

print("Features containing -1:", placeholder_review.height)

print("\nFeatures with over 50% -1:")
display(
    placeholder_review.filter(
        pl.col("minus_one_pct") > 50
    )
)

print("\nAll affected features:")
pl.Config.set_tbl_rows(100)
display(placeholder_review)

Features containing -1: 81

Features with over 50% -1:


feature,minus_one_count,minus_one_pct,missing_pct
str,i64,f64,f64
"""deferral_term""",2781854,95.09,0.0
"""hardship_length""",2781854,95.09,0.0
"""hardship_dpd""",2781855,95.09,0.0
"""orig_projected_additional_accr…",2746252,93.87,0.0
"""hardship_amount""",2743416,93.78,0.0
…,…,…,…
"""mths_since_last_record""",2498044,85.39,0.0
"""mths_since_recent_bc_dlq""",2274850,77.76,0.0
"""mths_since_last_major_derog""",2202575,75.29,0.0



All affected features:


feature,minus_one_count,minus_one_pct,missing_pct
str,i64,f64,f64
"""deferral_term""",2781854,95.09,0.0
"""hardship_length""",2781854,95.09,0.0
"""hardship_dpd""",2781855,95.09,0.0
"""orig_projected_additional_accr…",2746252,93.87,0.0
"""hardship_amount""",2743416,93.78,0.0
"""hardship_payoff_balance_amount""",2743416,93.78,0.0
"""hardship_last_payment_amount""",2743416,93.78,0.0
"""sec_app_revol_util""",2730905,93.35,0.0
"""revol_bal_joint""",2727669,93.24,0.0


## 3. Loan outcomes, years and terms

**Why:** Check whether our binary target is sufficiently observed in different loan vintages and in 36- versus 60-month loans. Later loans may still be unresolved.

**Output:** overall status counts and a single reusable `year_term_status` summary (`00_year_term_status.csv`). The original dataset is not filtered.

In [9]:
# issue_d is Datetime in the inspected Feather file; term is numeric.
loans = lf.select(
    pl.col("issue_d").dt.year().alias("issue_year"),
    pl.col("term").cast(pl.Int16, strict=False).alias("term_months"),
    pl.col("loan_status").cast(pl.String).str.strip_chars()
        .str.to_lowercase().fill_null("<missing>").alias("status")
)

year_term_status = (
    loans.group_by("issue_year", "term_months", "status")
    .agg(pl.len().alias("count"))
    .collect(engine="streaming")
)
status_summary = (
    year_term_status.group_by("status")
    .agg(pl.col("count").sum().alias("loans"))
    .with_columns((pl.col("loans") * 100 / n_rows).round(2).alias("pct"))
    .sort("loans", descending=True)
)
resolved = status_summary.filter(pl.col("status").is_in(["fully paid", "charged off"]))
print("All loan statuses:")
display(status_summary)
print("Strict resolved outcomes:", int(resolved["loans"].sum()))
year_term_status.write_csv(AUDIT_DIR / "00_year_term_status.csv")


All loan statuses:


status,loans,pct
str,u32,f64
"""fully paid""",1497783,51.2
"""current""",1031016,35.24
"""charged off""",362548,12.39
"""late (31-120 days)""",16154,0.55
"""in grace period""",10028,0.34
"""late (16-30 days)""",2719,0.09
"""issued""",2062,0.07
"""does not meet the credit polic…",1988,0.07
"""does not meet the credit polic…",761,0.03


Strict resolved outcomes: 1860331


## 4. One preliminary feature registry

**Why:** Separate fields that are unusable at prediction time from fields worth exploring. We should not discard a feature merely because of its full-archive missingness or weak apparent correlation.

**Output:** `00_feature_registry.csv` contains all 141 features and their audit results. Categories are **review labels**, not an approved final feature list.

In [10]:
post_loan = {
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee", "last_pymnt_d",
    "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
    "last_fico_range_low", "last_fico_range_high", "pymnt_plan",
    "debt_settlement_flag", "debt_settlement_flag_date",
    "settlement_status", "settlement_date", "settlement_amount",
    "settlement_percentage", "settlement_term", "deferral_term",
    "payment_plan_start_date", "orig_projected_additional_accrued_interest"
}
ids = {"id", "member_id", "url", "Unnamed: 0"}
lender = {"grade", "sub_grade", "int_rate", "installment",
          "funded_amnt", "funded_amnt_inv"}
joint = {"application_type", "annual_inc_joint", "dti_joint",
         "verification_status_joint", "revol_bal_joint"}

def category(name):
    if name == "loan_status": return "TARGET"
    if name == "issue_d": return "DATE_METADATA"
    if name in ids: return "IDENTIFIER"
    if name in post_loan or name.startswith("hardship_"): return "POST_LOAN_EXCLUDE"
    if name in lender: return "LENDER_REVIEW"
    if name in joint or name.startswith("sec_app_"): return "JOINT_REVIEW"
    return "PREDICTOR_REVIEW"

registry = audit.with_columns(
    pl.Series("role", [category(c) for c in audit["feature"].to_list()]),
    pl.lit("PENDING").alias("decision")
)
print("Provisional feature groups:")
display(registry.group_by("role").len().sort("len", descending=True))
registry.write_csv(AUDIT_DIR / "00_feature_registry.csv")


Provisional feature groups:


role,len
str,u32
"""PREDICTOR_REVIEW""",83
"""POST_LOAN_EXCLUDE""",32
"""JOINT_REVIEW""",16
"""LENDER_REVIEW""",6
"""IDENTIFIER""",2
"""DATE_METADATA""",1
"""TARGET""",1


## 5. Year-wise availability of rollout-sensitive features

**Why:** Some credit-bureau features appeared much later than others. Looking only at overall missing percentages would wrongly favor or reject entire feature groups.

**Output:** `00_feature_availability_by_year.csv` for a small set of known rollout-sensitive columns. This is an **initial targeted check**, not a final 12-feature restriction. For these columns, we provisionally count `-1` as unavailable; validate the meaning before cleaning.

In [11]:
rollout_features = [
    "bc_util", "mort_acc", "tot_cur_bal", "pct_tl_nvr_dlq",
    "avg_cur_bal", "open_acc_6m", "inq_last_12m", "all_util",
    "il_util", "mths_since_last_delinq", "mths_since_last_record",
    "dti_joint"
]
rollout_features = [c for c in rollout_features if c in columns]

coverage = (
    lf.with_columns(pl.col("issue_d").dt.year().alias("issue_year"))
    .group_by("issue_year")
    .agg(
        pl.len().alias("loans"),
        *[
            ((pl.col(c).is_not_null() & (pl.col(c) != -1).fill_null(False))
             .cast(pl.Float64).mean() * 100).round(2).alias(c)
            for c in rollout_features
        ]
    )
    .sort("issue_year")
    .collect(engine="streaming")
)
display(coverage.filter(pl.col("issue_year").is_between(2012, 2017)))
coverage.write_csv(AUDIT_DIR / "00_feature_availability_by_year.csv")


issue_year,loans,bc_util,mort_acc,tot_cur_bal,pct_tl_nvr_dlq,avg_cur_bal,open_acc_6m,inq_last_12m,all_util,il_util,mths_since_last_delinq,mths_since_last_record,dti_joint
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2012,53367,84.92,85.96,48.02,48.02,48.02,0.0,0.0,0.0,0.0,41.16,2.76,0.0
2013,134814,99.21,100.0,100.0,99.89,100.0,0.0,0.0,0.0,0.0,43.5,11.87,0.0
2014,235629,98.89,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0,50.82,17.62,0.0
2015,421095,99.0,100.0,100.0,100.0,100.0,5.08,5.08,5.08,4.42,51.56,17.67,0.12
2016,434407,98.89,100.0,100.0,100.0,100.0,99.99,99.99,99.98,86.78,52.85,19.09,2.02
2017,443579,98.66,100.0,100.0,100.0,100.0,100.0,100.0,99.98,86.07,49.8,17.18,9.56


## 6. Compare candidate cohorts (no final split yet)

**Why:** A cohort must balance sample size, available predictors, and reliable outcomes. We will compare 36- and 60-month terms rather than assume all resolved loans have equal follow-up.

**Output:** a small year/term comparison and totals for the **provisional** 2013–2016 / 36-month cohort. Do **not** filter and save a final training dataset yet; the snapshot date, maturity assumptions, and feature eligibility remain under review. Splitting happens in Notebook 02.

In [12]:
term_comparison = (
    year_term_status.filter(pl.col("issue_year").is_between(2012, 2017))
    .group_by("issue_year", "term_months")
    .agg(
        pl.col("count").sum().alias("total"),
        pl.col("count").filter(pl.col("status") == "fully paid")
            .sum().alias("fully_paid"),
        pl.col("count").filter(pl.col("status") == "charged off")
            .sum().alias("charged_off")
    )
    .with_columns((pl.col("fully_paid") + pl.col("charged_off")).alias("resolved"))
    .with_columns((pl.col("resolved") * 100 / pl.col("total"))
                  .round(2).alias("resolved_pct"))
    .sort("issue_year", "term_months")
)
display(term_comparison)

candidate = term_comparison.filter(
    pl.col("issue_year").is_between(2013, 2016)
    & (pl.col("term_months") == 36)
)
print("\nProvisional 2013–2016, 36-month cohort:")
print("All loans:", int(candidate["total"].sum()))
print("Resolved:", int(candidate["resolved"].sum()))
print("Charged off:", int(candidate["charged_off"].sum()))


issue_year,term_months,total,fully_paid,charged_off,resolved,resolved_pct
i32,i16,u32,u32,u32,u32,f64
2012,36,43470,37567,5903,43470,100.0
2012,60,9897,7156,2741,9897,100.0
2013,36,100422,88044,12378,100422,100.0
2013,60,34392,25743,8649,34392,100.0
2014,36,162570,140255,22315,162570,100.0
…,…,…,…,…,…,…
2015,60,137922,82130,37516,119646,86.75
2016,36,323495,272076,51299,323375,99.96
2016,60,110912,51946,27711,79657,71.82



Provisional 2013–2016, 36-month cohort:
All loans: 869660
Resolved: 869540
Charged off: 128170


## 7. Cohort Finalization and Interim Dataset

**Objective:** Select a suitable working population for predicting loan charge-off and create a reproducible interim dataset.

**Cohort selection:** Loans issued during 2013–2016, with a 36-month term and a resolved status of either Fully Paid or Charged Off.

**Justification:** This period offers a substantial sample size, nearly complete observed resolution, and good coverage of established credit-history features. Later periods contain more unresolved loans, while several additional credit features are unavailable in earlier years.

**Target:** Fully Paid = 0; Charged Off = 1.

**Output:** A Parquet dataset containing the selected loans, original columns, and binary target.

**Limitations:** This is a provisional modeling cohort. Resolved-outcome selection, data-snapshot timing, and exclusion of 60-month loans must be acknowledged. The dataset is not yet preprocessed or split into training and testing sets.

In [15]:
# 6 — Finalize working cohort and save interim dataset

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

output_path = ROOT / "data/interim/lendingclub_2013_2016_36m.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)

total = 0
charged_off = 0
writer = None

try:
    with pa.memory_map(str(DATA_PATH), "r") as source:
        reader = pa.ipc.open_file(source)

        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)

            # Decode categorical columns safely.
            arrays = [
                pc.dictionary_decode(col)
                if pa.types.is_dictionary(col.type)
                else col
                for col in batch.columns
            ]

            batch = pa.RecordBatch.from_arrays(
                arrays, names=batch.schema.names
            )

            # Get essential fields.
            issue_d = batch.column("issue_d")
            term = batch.column("term")

            status = pc.utf8_lower(
                pc.utf8_trim_whitespace(
                    pc.cast(batch.column("loan_status"), pa.string())
                )
            )

            year = pc.year(issue_d)

            # Select working cohort.
            mask = (
                pc.and_(
                    pc.greater_equal(year, 2013),
                    pc.less_equal(year, 2016)
                )
            )

            mask = pc.and_(
                mask,
                pc.equal(term, 36)
            )

            mask = pc.and_(
                mask,
                pc.is_in(
                    status,
                    value_set=pa.array(
                        ["fully paid", "charged off"]
                    )
                )
            )

            selected = batch.filter(mask)

            if selected.num_rows == 0:
                continue

            # Binary target
            selected_status = pc.utf8_lower(
                pc.utf8_trim_whitespace(
                    pc.cast(
                        selected.column("loan_status"),
                        pa.string()
                    )
                )
            )

            target = pc.cast(
                pc.equal(selected_status, "charged off"),
                pa.int8()
            )

            selected = pa.RecordBatch.from_arrays(
                list(selected.columns) + [target],
                names=list(selected.schema.names) + ["target"]
            )

            # Save incrementally.
            if writer is None:
                writer = pq.ParquetWriter(
                    str(output_path),
                    selected.schema,
                    compression="zstd"
                )

            writer.write_batch(selected)

            total += selected.num_rows
            charged_off += pc.sum(target).as_py() or 0

finally:
    if writer is not None:
        writer.close()

print("INTERIM DATASET CREATED")
print("Rows:", f"{total:,}")
print("Fully Paid:", f"{total - charged_off:,}")
print("Charged Off:", f"{charged_off:,}")
print("Saved:", output_path)

# Verify against our previous cohort audit.
if total != 869_540 or charged_off != 128_170:
    print("WARNING: Counts differ from the previous audit.")
    print("Review the cohort before using this file.")
else:
    print("PASS: Cohort counts match previous results.")

INTERIM DATASET CREATED
Rows: 869,540
Fully Paid: 741,370
Charged Off: 128,170
Saved: d:\ML_Projects\P2P-Lending-Default-Prediction\data\interim\lendingclub_2013_2016_36m.parquet
PASS: Cohort counts match previous results.


## Decision checkpoint — Notebook 00

**Known from earlier runs:** the 2013–2016 / 36-month candidate contains approximately **869,660 total** and **869,540 strictly resolved** loans. These figures should match a fresh run. The observed resolved percentage does **not** prove complete lifetime outcomes; the archive's snapshot date needs independent confirmation.

**Keep for now:** raw Feather unchanged, a single column audit, a preliminary feature registry, a focused availability table, and the 36/60-month cohort comparison.

**Do not do here:** automatic feature elimination based on missingness, imputation, outlier removal, target-based feature selection, scaling, model training or data splitting.

**Next:** Notebook 01 — focused EDA and feature-engineering hypotheses. Notebook 02 — chronological split, then train-only fitted preprocessing. A GUI will later use the exact saved preprocessing + model pipeline.